# 🔍 Workforce Attrition Analytics — Exploratory Analysis

This notebook explores the synthetic HR dataset to understand:
1. Attrition distribution across departments and job levels
2. Behavioral telemetry patterns (overtime, absences)
3. Engagement survey score distributions
4. Correlation between features and attrition
5. Burnout hotspot identification

> **Note:** Run `python database/seed_data.py` and `python -m etl.ingest` first to generate data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

%matplotlib inline
print('Libraries loaded ✓')

In [ ]:
# Load data
data_dir = Path('../data/raw')

df_emp    = pd.read_parquet(data_dir / 'employees.parquet')
df_attr   = pd.read_parquet(data_dir / 'attrition.parquet')
df_tel    = pd.read_parquet(data_dir / 'telemetry.parquet')
df_surv   = pd.read_parquet(data_dir / 'surveys.parquet')
df_perf   = pd.read_parquet(data_dir / 'performance.parquet')

print(f'Employees: {len(df_emp):,}')
print(f'Telemetry records: {len(df_tel):,}')
print(f'Survey records: {len(df_surv):,}')
print(f'Performance records: {len(df_perf):,}')

# Merge attrition flag into employee base
df_attr['left_company_flag'] = df_attr['left_company_flag'].astype(int)
df = df_emp.merge(df_attr[['emp_id', 'left_company_flag']], on='emp_id', how='left')
df['left_company_flag'] = df['left_company_flag'].fillna(0).astype(int)
print(f'\nAttrition rate: {df["left_company_flag"].mean():.1%}')

In [ ]:
# ── 1. Attrition by Department ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

dept_attrition = df.groupby('department').agg(
    total=('emp_id', 'count'),
    left=('left_company_flag', 'sum')
).assign(attrition_rate=lambda x: x['left'] / x['total'] * 100).sort_values('attrition_rate', ascending=False)

dept_attrition['attrition_rate'].plot(kind='barh', ax=axes[0], color='#E63946')
axes[0].set_title('Attrition Rate by Department (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Attrition Rate (%)')

# Job Level
level_attrition = df.groupby('job_level').agg(
    total=('emp_id', 'count'),
    left=('left_company_flag', 'sum')
).assign(attrition_rate=lambda x: x['left'] / x['total'] * 100)

level_order = ['Junior', 'Mid', 'Senior', 'Lead', 'Manager', 'Director']
level_attrition = level_attrition.reindex([l for l in level_order if l in level_attrition.index])
level_attrition['attrition_rate'].plot(kind='bar', ax=axes[1], color='#457B9D', rot=0)
axes[1].set_title('Attrition Rate by Job Level (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Attrition Rate (%)')

plt.tight_layout()
plt.savefig('../reports/attrition_by_dept_level.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2. Overtime Burnout Heatmap ─────────────────────────────────────────────
df_tel['log_date'] = pd.to_datetime(df_tel['log_date'])
df_tel['month'] = df_tel['log_date'].dt.to_period('M').astype(str)

tel_with_dept = df_tel.merge(df_emp[['emp_id', 'department']], on='emp_id', how='left')
pivot = tel_with_dept.groupby(['department', 'month'])['overtime_hours'].mean().unstack(fill_value=0)

# Keep last 12 months
pivot = pivot.iloc[:, -12:]

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    pivot, cmap='YlOrRd', annot=True, fmt='.1f',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Overtime Hours/Day'}
)
ax.set_title('Burnout Hotspot Matrix — Dept × Month (Avg Overtime Hours)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Department')
plt.tight_layout()
plt.savefig('../reports/burnout_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3. Engagement Score Distribution ────────────────────────────────────────
score_cols = ['satisfaction_score', 'work_life_balance_score', 'manager_relationship_score',
              'career_growth_score', 'compensation_fairness_score', 'team_cohesion_score']

latest_survey = df_surv.sort_values('survey_date').groupby('emp_id').last().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    axes[i].hist(latest_survey[col].dropna(), bins=20, color='#1D3557', alpha=0.8, edgecolor='white')
    axes[i].axvline(latest_survey[col].mean(), color='#E63946', linestyle='--', lw=2, label=f'Mean: {latest_survey[col].mean():.2f}')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11)
    axes[i].set_xlabel('Score (1-5)')
    axes[i].legend(fontsize=9)

plt.suptitle('Engagement Survey Score Distributions', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/engagement_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4. Feature Correlation with Attrition ───────────────────────────────────
from etl.features import generate_workforce_features

master_df = generate_workforce_features(df_emp, df_tel, latest_survey, 
                                         df_perf.sort_values('review_date').groupby('emp_id').last().reset_index())
master_df = master_df.merge(df_attr[['emp_id', 'left_company_flag']], on='emp_id', how='left')
master_df['left_company_flag'] = master_df['left_company_flag'].fillna(0).astype(int)

numeric_features = [
    'tenure_years', 'base_salary', 'avg_overtime_90d', 'absence_trend_90d',
    'satisfaction_score', 'work_life_balance_score', 'career_growth_score',
    'compensation_fairness_score', 'performance_rating', 'promotions_count',
    'burnout_risk_score', 'composite_engagement_score'
]

corr = master_df[numeric_features + ['left_company_flag']].corr()['left_company_flag'].drop('left_company_flag').sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E63946' if v > 0 else '#457B9D' for v in corr.values]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Feature Correlation with Attrition (Pearson)', fontsize=14, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.savefig('../reports/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop attrition drivers (positive correlation = increases risk):')
print(corr.tail(5).to_string())
print('\nTop retention factors (negative correlation = reduces risk):')
print(corr.head(5).to_string())

## ✅ Next Steps
1. Run `python models/train_pipeline.py` to train the XGBoost classifier
2. Check `reports/roc_curve.png` and `reports/shap_summary.png` for model evaluation
3. Check `data/processed/risk_scores.csv` for per-employee flight risk scores
4. Configure Apache Superset with `dashboard/superset_config.py` to build the BI layer